# 03: height, velocity and uncertainty

E3-E7 take 145 minutes. Run setup in a fresh kernel; results go to `data/<run_id>/`.

| Exercise | Your work | Supplied support and data |
| --- | --- | --- |
| E3 | Four filter expressions; choose Q/V. | Model, checks and plots; live/saved acceleration and ToF, or simulation. |
| E4 | Chi-square threshold and rejection probability. | Injected outlier and matched replay using your filter. |
| E5 | Choose gap/rest intervals and uncertainty; derive the velocity constraint. | E3 replay for range loss; simulation and supplied update for valid/false rest. |
| E6 | Predict and assess a frozen filter on independent input. | Fresh live/saved data or a labelled independent simulation. |
| E7 | Clearance-to-speed rule and margins. | E3/E5 covariance, illustrative gates and assumed horizontal uncertainty. |

Live acquisition is motor-free. Simulation provides truth for accuracy/coverage checks; fresh measured data test transfer to the drone.


In [ ]:
from pathlib import Path
import os
import sys
from dataclasses import replace

# Shared helpers are beside day2; data stays beside these notebooks.
notebook_dir = Path.cwd()
if (notebook_dir / "code/day2").is_dir():
    notebook_dir = notebook_dir / "code/day2"
sys.path.insert(0, str(notebook_dir.resolve().parent))

import numpy as np
from IPython.display import Image, display
from day2.notebook_tools import capture_vertical, display_table, next_attempt, drone_uri

run_id = "trial-01"  # Label this session; each comparison gets a fresh attempt folder.
seed = 0  # Reproducible simulation fallback and matched comparisons.
output = notebook_dir / "data" / run_id
output.mkdir(parents=True, exist_ok=True)
print("Results:", output.resolve())

LABEL = None  # Set the sticker label for live data: 2 or "A", as on Day 1.
uri = drone_uri(LABEL) if LABEL is not None else os.environ.get("CF_URI", "")

from day2 import vertical


## From tracking to drone sensor fusion

Day 1 used onboard position and velocity for feedback. Here we estimate height and velocity from sensors. Adapt E1: detector distance becomes time, position/slope becomes height/velocity, and the gap becomes recorded dt. The gain and measurement update stay the same; acceleration enters through B, with interval uncertainty Q. The accelerometer supplies the process input; the downward ToF sensor measures distance to a fixed floor. A level drone moving gently up and down makes range approximately height. Apply E2’s calibration method: subtract the initial stationary `acc.z` mean, including gravity, and multiply by 9.80665 for approximate vertical acceleration; use the rest range spread to estimate V. This retains existing onboard sensor compensation and uses no onboard position, velocity or attitude estimate.

Both channels carry firmware log timestamps in the same packet. The helper unwraps milliseconds to seconds, uses actual dt, and rejects duplicate/reversed times or gaps above 0.30 s. Log timestamps approximate acquisition time; channel delays and internal filtering remain model limitations. ToF range is floor-relative, rather than Lighthouse/world height.


## Capture one sensor recording, or use simulation

A Flow deck with a working downward ToF sensor is needed for live data; Lighthouse positioning is not used. Close cfclient and other radio clients. Place the drone on a stable, level surface with a clear view of a fixed flat floor and keep hands clear of the propellers. Set the assigned URI and opt in below.

Use a stable raised support with an opening beneath the sensor, 5-20 cm above the floor. The ToF beam must see the same fixed floor during the initial rest and subsequent motion.

The helper sets `stabilizer.stop=1`, verifies the firmware lock and zero motor outputs, and keeps monitoring them. **Leave the drone on the surface until STOP VERIFIED.** Then keep it stationary and level for five seconds. After **Calibration check passed**, move it gently up and down while keeping the beam on the same floor. Hold still again by the 15-second prompt. Capture ends at 20 seconds; note which rest intervals actually occurred.

The stop stays engaged after success, interruption or failure. Place the drone down and power-cycle before normal flight/preflight checks. Connection plus capture is bounded to 50 seconds. If acquisition or model checks stall, a checked sensor recording or simulation lets you continue; spend at most five minutes troubleshooting and retain unexpected data for later investigation.


In [ ]:
run_live_capture = False
ready_for_motor_stop = False  # True after placing the drone down and closing radio clients.
capture_name = "capture-01"  # Choose a new name for another recording.
captured_directory = None  # Invalidate any previous capture before a new attempt.
if run_live_capture:
    data = settings = training_calibration = None
    baseline_result = dropout = validation_data = validation_result = None
    captured_directory = capture_vertical(
        output / capture_name, uri, ready_for_motor_stop=ready_for_motor_stop, duration_s=20)
else:
    print("Capture skipped. Explicitly choose saved sensor data or simulation below.")


## Choose data once

Set `recording_directory` to a complete sensor recording containing `raw.csv` and `recording.json`, or explicitly leave it `None` for the simulated rest/up/down/rest experiment. This format has timestamped acceleration and range, not onboard position estimates. Files are hash-checked; invalid selections are errors, not silent fallback.

Selection clears the previous dataset before loading. The five-second rest calibration precedes filtering, so later motion never contributes to its fitted offset. All E3/E4/dropout comparisons reuse the same input and calibration. Simulation supplies truth for RMSE/coverage; a real recording has no independent truth.


In [ ]:
data = None
baseline_result = None
dropout = None
validation_result = None
settings = None
training_calibration = None
recording_directory = captured_directory  # Or: root / "data/vertical-training"; None selects simulation.
data = vertical.select(recording_directory, seed=seed)
training_calibration = data.calibration
settings = vertical.Settings(range_std_m=training_calibration.range_std_m)
print("Source:", data.metadata["source"], "input hash:", data.metadata["input_sha256"])
print("Calibration:", training_calibration)
print("Filtering timestamps:", data.time_s[0], "to", data.time_s[-1], "seconds")


## E3: construct the filter, then change its assumptions — 50 minutes

Allow 25 minutes for data and four expressions (at most five on acquisition), 20 for comparisons and 5 for discussion. Logging and plotting are supplied.

Adapt E1 to $x=(z,v_z)^T$, $H=(1,0)$ and measured time intervals:
$$F=\begin{pmatrix}1&\Delta t\\0&1\end{pmatrix},\quad B=\begin{pmatrix}\tfrac12\Delta t^2\\\Delta t\end{pmatrix},\quad Q=BB^T\sigma_a^2.$$
$$x^-=Fx+Ba,\quad C^-=FCF^T+Q,\quad r=m-Hx^-,\quad R=HC^-H^T+V,$$
$$K=C^-H^T/R,\quad x^+=x^-+Kr.$$
Q represents independent acceleration perturbations held over each interval; its time dependence differs from continuous white-acceleration noise. The Joseph covariance update is supplied; `vertical.predict` and `vertical.update` provide worked references.

Complete the four `None` expressions using E1’s structure plus acceleration input. The checks run before replay. For one positive height residual, trace K, state corrections and covariance. Velocity changes through $K_v=C^-_{vz}/R$ (units 1/s). Repeat with cross-covariance zero, then a different dt and V; explain the changes.

Compare acceleration integration, range differentiation and the KF on identical timestamps. Vary V by 0.1 and 10; choose one acceleration standard deviation for Q. Keep measurements and calibration fixed. Inspect height, velocity, ±1.96-sigma bands and **pre-gate** pull widths. A consistent unbiased Gaussian model predicts unit pull width and **95%** marginal coverage. Simulation provides truth; real-data accuracy needs an independent reference.

**Completion:** explain one update and the velocity correction; keep one V comparison and a justified Q choice.


In [ ]:
def our_predict(x, C, acceleration, dt, acceleration_std):
    F = np.array([[1., dt], [0., 1.]])
    B = np.array([0.5 * dt**2, dt])
    Q = np.outer(B, B) * acceleration_std**2
    x_predicted = None  # TODO E3: propagate the state.
    C_predicted = None  # TODO E3: propagate covariance and add Q.
    if x_predicted is None or C_predicted is None:
        raise NotImplementedError("Complete the two prediction expressions.")
    return x_predicted, C_predicted

def our_update(x_predicted, C_predicted, measurement, V):
    H = np.array([1., 0.])
    residual = measurement - H @ x_predicted
    R = H @ C_predicted @ H + V
    K = None  # TODO E3: gain from covariance and residual variance.
    x_filtered = None  # TODO E3: correct the predicted state.
    if K is None or x_filtered is None:
        raise NotImplementedError("Complete the gain and state-correction expressions.")
    A = np.eye(2) - np.outer(K, H)
    C_filtered = A @ C_predicted @ A.T + np.outer(K, K) * V
    return x_filtered, C_filtered

vertical.check_steps(our_predict, our_update)


In [ ]:
x = np.array([0.3, 0.0])
C = np.diag([0.02**2, 0.10**2])
dt = data.time_s[1] - data.time_s[0]
x_predicted, C_predicted = our_predict(x, C, data.acceleration_m_s2[1], dt,
                                      settings.acceleration_std_m_s2)
measurement = x_predicted[0] + 0.01  # A specified +1 cm predicted residual.
K = C_predicted[:, 0] / (C_predicted[0, 0] + settings.range_std_m**2)
x_filtered, C_filtered = our_update(x_predicted, C_predicted, measurement, settings.range_std_m**2)
print("dt [s], K [dimensionless, 1/s]:", dt, K)
print("Correction [m, m/s]:", x_filtered - x_predicted)
print("C before/after:\n", C_predicted, "\n", C_filtered)


In [ ]:
baseline_settings = vertical.Settings(range_std_m=training_calibration.range_std_m)
candidate_acceleration_std = 0.08  # Choose and justify; observations stay fixed.
chosen_name = "Q-candidate"  # Select one matched case after inspecting the comparison.


In [ ]:
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
baseline_result = vertical.run(data, baseline_settings, predict_step=our_predict, update_step=our_update)
cases = {
    "baseline": baseline_result,
    "V-small": vertical.run(data, replace(baseline_settings, range_std_m=baseline_settings.range_std_m * np.sqrt(0.1)),
                            predict_step=our_predict, update_step=our_update),
    "V-large": vertical.run(data, replace(baseline_settings, range_std_m=baseline_settings.range_std_m * np.sqrt(10)), predict_step=our_predict, update_step=our_update),
    "Q-candidate": vertical.run(data, replace(baseline_settings, acceleration_std_m_s2=candidate_acceleration_std), predict_step=our_predict, update_step=our_update),
}
e3_output = next_attempt(output / "E3")
display(Image(filename=str(vertical.save_comparison(data, cases, e3_output))))
display_table([dict(case=name, **vertical.summary(data, result)) for name, result in cases.items()])
settings = vertical.Settings(**cases[chosen_name]["settings"])
baseline_result = cases[chosen_name]
print("Frozen training choice:", chosen_name, settings)


## E4: one anomalous measurement — 25 minutes

A scalar height predicted residual has one degree of freedom. Replace `None` in `our_residual_threshold` using `chi2.ppf(1-alpha, df=dimension)`, and explain the chosen false-rejection probability. The default alpha is 0.01. The acceptance function below tests $r^2/R$ against that threshold before applying the range update.

Predict the state correction from a **0.15 m** added range observation at/after **12 s**, then compare ungated and gated runs on that identical copy. Inspect the saved residual, its variance, chi-square, prospective gain, acceptance and recovery. The original recording is unchanged.

A large chi-square can also indicate a poor propagation model, timing error or changed floor geometry. Rejection is not a substitute for understanding those cases. Count rejections on clean data as well; retain rejected residuals in the diagnostic distribution.

**Completion:** one explained rejection and one reason a valid measurement might be rejected. More fault scenarios are optional.


In [ ]:
from scipy.stats import chi2

false_rejection_probability = 0.01  # Choose and justify.

def our_residual_threshold(dimension, alpha):
    # TODO E4: replace None with the dimension-aware threshold.
    return None


def our_accept_range(residual, R, threshold):
    chi_squared = residual**2 / R
    return threshold is None or chi_squared <= threshold


In [ ]:
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
fault_data = vertical.with_outlier(data, time_s=12, offset_m=0.15)
ungated = vertical.run(fault_data, settings, predict_step=our_predict, update_step=our_update)
gated = vertical.run(fault_data, settings, predict_step=our_predict, update_step=our_update,
                     residual_gate=our_residual_threshold, gate_step=our_accept_range, alpha=false_rejection_probability)
clean_gated = vertical.run(data, settings, predict_step=our_predict, update_step=our_update,
                           residual_gate=our_residual_threshold, gate_step=our_accept_range,
                           alpha=false_rejection_probability)
print("Clean:", vertical.summary(data, clean_gated))
print("Fault, gated:", vertical.summary(fault_data, gated))
e4_clean_output = next_attempt(output / "E4-clean")
display(Image(filename=str(vertical.save_comparison(
    data, {"ungated": baseline_result, "gated": clean_gated}, e4_clean_output))))
e4_output = next_attempt(output / "E4")
display(Image(filename=str(vertical.save_comparison(
    fault_data, {"ungated": ungated, "gated": gated}, e4_output))))


## E5: missing range and a rest constraint — 30 minutes

**First 15 minutes:** suppress range updates during **8-11 s** in replay, retaining acceleration. Predict and compare uncertainty, covariance and gain before, during and after the gap, including the correction when range returns.

**Next 15 minutes:** derive a known-rest measurement $m=0$, $H=(0,1)$ with finite variance. Choose its standard deviation and an interval within simulated rest at **15-20 s**. Compare no constraint, valid rest and deliberately false rest during motion. Quiet IMU readings alone also permit constant-speed translation.

All three runs reuse your E3 functions. Trace the supplied zero-velocity update: velocity retains finite uncertainty, and height also changes through cross-covariance.

**Completion:** explain uncertainty growth and the effect of valid/false constraints. Automatic stationary detection using `duration_s` and fresh `flow_speed_m_s` evidence is optional.


In [ ]:
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
if baseline_result is None:
    raise ValueError("Rerun E3 on the selected data before continuing.")
range_gap = (8.0, 11.0)
dropout = vertical.run(data, settings, predict_step=our_predict, update_step=our_update, range_gap=range_gap)
e5_dropout_output = next_attempt(output / "E5-dropout")
display(Image(filename=str(vertical.save_comparison(
    data, {"baseline": baseline_result, "range-gap": dropout}, e5_dropout_output))))
gap_index = np.flatnonzero(data.time_s <= range_gap[1])[-1]
print("At gap end, baseline C:\n", baseline_result["covariance"][gap_index])
print("At gap end, dropout C:\n", dropout["covariance"][gap_index])
recovery_index = np.flatnonzero(dropout["accepted"] & (data.time_s > range_gap[1]))[0]
print("First returning range at [s]:", data.time_s[recovery_index])
print("Predicted residual, variance, gain:", dropout["predicted_residual"][recovery_index],
      dropout["predicted_residual_variance"][recovery_index], dropout["gain"][recovery_index])


In [ ]:
rest_settings = replace(settings, stationary_velocity_std_m_s=0.02)  # Choose and justify.
known_rest = [(16.0, 19.0)]
false_rest = [(8.0, 9.0)]  # Deliberate simulation-only counterexample.

# Treat known rest as a velocity measurement of zero with finite variance V.
def our_zero_velocity_update(x, C, V):
    H = np.array([0., 1.])
    residual = -H @ x
    R = H @ C @ H + V
    K = C @ H / R
    x_filtered = x + K * residual
    A = np.eye(2) - np.outer(K, H)
    C_filtered = A @ C @ A.T + np.outer(K, K) * V
    return x_filtered, C_filtered


In [ ]:
rest_data = vertical.select(seed=seed)  # Known simulated rest; truth never enters the update.
rest_runs = {
    "without-constraint": vertical.run(rest_data, rest_settings, predict_step=our_predict, update_step=our_update),
    "known-rest": vertical.run(rest_data, rest_settings, predict_step=our_predict, update_step=our_update, stationary_step=our_zero_velocity_update, stationary_intervals=known_rest),
    "false-rest": vertical.run(rest_data, rest_settings, predict_step=our_predict, update_step=our_update, stationary_step=our_zero_velocity_update, stationary_intervals=false_rest),
}
e5_rest_output = next_attempt(output / "E5-rest")
display(Image(filename=str(vertical.save_comparison(rest_data, rest_runs, e5_rest_output))))
for name, result in rest_runs.items():
    print(name, vertical.summary(rest_data, result))


## E6: frozen settings on fresh data — 15 minutes

Spend five minutes selecting data, five comparing and five discussing. Freeze E3's Q/V and rest calibration. For live acquisition, set `run_validation_capture` and confirm `ready_for_validation_motor_stop`; reuse the handling checks and choose a fresh output name. Spend at most five minutes troubleshooting. `validation_directory=None` selects labelled simulation with independent noise (seed +20).

Identical input hashes are rejected. Capture verifies rest; filtering retains the training correction, timestamps and 0.30 s gap limit. Initial height comes from the new recording, with configured velocity uncertainty. Explain startup differences before comparing later behaviour.

Simulation uses frozen sensor assumptions to test implementation consistency; fresh measured data test their transfer to the drone.

**Completion:** check one prediction, give one limitation and assess the frozen assumptions. Mark experimental validation outstanding if using simulation alone.


In [ ]:
validation_data = None
validation_result = None
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
validation_data = None  # A failed selection must not leave the previous validation input active.
validation_directory = None  # A distinct capture/supplied sensor directory, or explicit simulation.
run_validation_capture = False
ready_for_validation_motor_stop = False  # Confirm the drone is down, hands clear and radio clients closed.
if run_validation_capture:
    validation_directory = capture_vertical(
        output / "validation-capture-01", uri, duration_s=20,
        ready_for_motor_stop=ready_for_validation_motor_stop)
validation_data = vertical.select(validation_directory, seed=seed + 20, calibration=training_calibration, settings=settings)
if validation_data.metadata["input_sha256"] == data.metadata["input_sha256"]:
    validation_data = None
    raise ValueError("Choose a different training/validation recording.")
validation_result = vertical.run(validation_data, settings, predict_step=our_predict, update_step=our_update)
e6_output = next_attempt(output / "E6")
display(Image(filename=str(vertical.save_comparison(
    validation_data, {"frozen-settings": validation_result}, e6_output))))
print("Validation source:", validation_data.metadata["source"])
print(vertical.summary(validation_data, validation_result))


## E7: use the filter's covariance in a race decision — 25 minutes

Take **actual C** at the same timestamp from E3 and the E5 range-gap run. The height variance C[0,0] supplies the vertical entry of the three-dimensional position covariance below. Horizontal uncertainties are explicitly assumed for this 1D experiment; it has not estimated X/Y motion.

Project with $C_{plane}=U^T C_{pp}U$, where U spans the gate plane. Compare normals [1,0,0] (vertical opening) and [0,0,1] (horizontal opening). Which opening is affected by growing height uncertainty? Uncertainty along the normal should not directly reduce in-plane clearance.

Clearance is opening radius minus radial offset, drone radius, tracking margin and k times the worst in-plane sigma. Defaults are illustrative; the sigma allowance is not an exact collision probability. Use the computed baseline/dropout values to justify a candidate speed rule, including **zero forward speed for non-positive clearance**. State the approach distance and braking assumptions before proposing physical use.

**Completion:** retain the two C projections and speed decisions, explaining the chain from missing measurement to uncertainty to action. These are planning decisions; speed recommendations do not change simulated motion or establish a lap-time improvement.

The example transfers uncertainty, rather than a floor-relative height value, into an illustrative gate geometry. Real clearance also needs floor/world reference and survey errors.


In [ ]:
if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")
if dropout is None:
    raise ValueError("Rerun E5's range-gap comparison before continuing.")
if baseline_result is None:
    raise ValueError("Rerun E3 on the selected data before continuing.")
from race_common.hoop import Hoop
from race_common.strategy import RaceLimits, recommended_speed

limits = RaceLimits()
estimated_position = np.array([1.0, 0.03, 0.7])
horizontal_std_m = 0.04  # Assumed here, not inferred by the vertical filter.

def our_crossing_speed(clearance, limits):
    if clearance <= 0:
        return 0.0
    # TODO E7: choose and justify a slower positive-clearance regime.
    return limits.nominal_speed_m_s

for label, result in (("range available", baseline_result), ("range gap", dropout)):
    C_position = np.diag([horizontal_std_m**2, horizontal_std_m**2,
                          result["covariance"][gap_index, 0, 0]])
    for name, normal in (("vertical", [1, 0, 0]), ("horizontal", [0, 0, 1])):
        gate = Hoop(np.array([1.0, 0.0, 0.7]), np.array(normal), radius_m=0.5)
        speed, clearance = recommended_speed(gate, estimated_position, C_position, limits)
        print(label, name, "projected covariance [m^2]:", gate.plane_covariance(C_position))
        print("Clearance [m]:", clearance, "baseline/candidate [m/s]:",
              speed, our_crossing_speed(clearance, limits))


## Discuss and keep

Keep the edited notebooks, calibration decision and one useful comparison. In your shared notes, explain one failed prediction and one race-day implication. Saved figures, CSVs and settings retain the detail; label data as live, supplied or simulated.

The [exercise guide](../../docs/kalman_filter_practical.md) summarizes the model and goals; [recording details](../../docs/sensor_recording_contract.md) cover units, timing and handling. Use the [glossary](../../docs/glossary.md) as needed.

### Optional background

Further theory and six-state studies are listed in the [day programme](../../docs/day_2_kalman.md#optional-background).
